In [ ]:
# goals:
# write down samples with the ACTUAL POS!!!! that was predicted!!!!
# then i can easily figure out how this should be analised 
# it would make a lot of sense to check what parts-of-speech are actually being predicted for +wh and -wh conditions
# across the different embedding levels etc, not just correctness

## Generate continuations

In [1]:
import h5py
import numpy as np
import spacy
from scipy.special import softmax
from utils import text_utils
import pandas as pd
import pickle

nlp = spacy.load("en_core_web_sm")

object_stimuli_path = "../data/stimuli/wilcox2022_embed4_object_gap.csv"
subject_stimuli_path = "../data/stimuli/wilcox2022_embed4_subject_gap.csv"

gpt2_probabilities_path_object = "../data/model_outputs/distributions/gpt2_wilcox2022_embed4_object_gap.hdf5"

grnn_probabilities_path_object = "../data/model_outputs/distributions/grnn_wilcox2022_embed4_object_gap.hdf5"

ngram_probabilities_path_object = "../data/model_outputs/distributions/ngram_wilcox2022_embed4_object_gap.hdf5"

gpt2_probabilities_path_subject = "../data/model_outputs/distributions/gpt2_wilcox2022_embed4_subject_gap.hdf5"

grnn_probabilities_path_subject = "../data/model_outputs/distributions/grnn_wilcox2022_embed4_subject_gap.hdf5"

ngram_probabilities_path_subject = "../data/model_outputs/distributions/ngram_wilcox2022_embed4_subject_gap.hdf5"

/home/marrsia/.local/lib/python3.8/site-packages/thinc/compat.py:36: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  hasattr(torch, "has_mps")
/home/marrsia/.local/lib/python3.8/site-packages/thinc/compat.py:37: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  and torch.has_mps  # type: ignore[attr-defined]


In [3]:
def get_cont_pos(sentence_starts, hdf5_path, n_samples=100):
    """
    For each sentence start, computes the probability of gap vs no_gap
    by sampling next tokens from an lm-zoo hdf5 file and applying the POS heuristic.
    
    Args:
        sentence_starts: dict of {original_idx: {sentence_start, condition, levels_of_embedding}}
        hdf5_path: path to hdf5 file of next-token predictions
        n_samples: number of tokens to sample from the distribution
    
    Returns:
        same dict with added keys: p_no_gap, p_gap, predicted
    """
   
    rows = []
    
    with h5py.File(hdf5_path, 'r') as f:
        vocab = [token.decode('utf-8') for token in f['vocabulary'][:]]
        original_indices = list(sentence_starts.keys())
        
        
        for seq_idx, original_idx in enumerate(original_indices):
            entry = sentence_starts[original_idx]
            entry["sentence_id"] = original_idx
            
            # getting sentence in case I want to print it for debugging
            sent = f['sentence'][str(seq_idx)]
            log_probs = sent['predictions'][-1]
            probs = softmax(log_probs).astype(float)
            probs = probs / probs.sum()
            
            sampled_indices = np.random.choice(len(probs), size=n_samples, p=probs)
            
            gap_count = 0
            no_gap_count = 0
            
            for token_idx in sampled_indices:
                token = vocab[token_idx].replace('Ġ', ' ').strip()
                doc = nlp(entry['sentence_start'] + ' ' + token)
                pos = doc[-1].pos_
                
                row = dict(entry)
                
                row['next_token_POS'] = pos
                
                rows.append(row)           
    df = pd.DataFrame(rows)
    return df

In [5]:
save_results_dir = "../data/model_outputs/distributions/cont_analysis_v2"

def compute_and_save_for_model(model_name, probabilities_path_object, probabilities_path_subject):
    sentence_starts_obj = text_utils.build_sentence_starts_for_sampling(object_stimuli_path, "object")
    sentence_starts_subj = text_utils.build_sentence_starts_for_sampling(subject_stimuli_path, "subject")

    results_obj = get_cont_pos(sentence_starts_obj, probabilities_path)

    results_obj.to_csv(save_results_dir + f"/{model_name}_object")

    results_subj = get_cont_pos(sentence_starts_subj, probabilities_path)

    results_subj.to_csv(save_results_dir + f"/{model_name}_subject")

In [6]:
compute_and_save_for_model("gpt2", gpt2_probabilities_path_object, gpt2_probabilities_path_subject)
compute_and_save_for_model("grnn", grnn_probabilities_path_object, grnn_probabilities_path_subject)
compute_and_save_for_model("ngram", ngram_probabilities_path_object, ngram_probabilities_path_subject)

### part-of-speech amounts

In [16]:
import pandas as pd

df = pd.read_csv('../data/model_outputs/distributions/cont_analysis_v2/ngram_object')

result = (
    df.groupby(['condition'])['next_token_POS']
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)


result.columns.name = None


result

,condition,ADJ,ADP,ADV,AUX,CCONJ,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,SPACE,SYM,VERB,X
0,+wh_gap,2469,172,1084,18,11,12,14898,1482,4,54,2554,66,13,10,3,3542,608
1,-wh_no_gap,2564,154,1040,11,0,9,14921,1648,6,39,2525,69,8,5,4,3378,619


## Mark for gap compatibility

In [17]:
OBJECT_GAP_INCOMPATIBLE = ['DET', 'PRON', 'PROPN']

SUBJECT_GAP_COMPATIBLE = ['VERB']